# Gateway

The only code that builds a model client. Everything else receives `get_chat_model` as an argument
and never learns what a gateway is.

Two base URLs: the AI Gateway serves the `finhive_router` and `finhive_embeddings` services, and
Databricks' own pay-per-token models answer on `/serving-endpoints`, which is where the guardrails
run because they are the most frequent and most mechanical calls in the graph (`agent-design.md`
§11.3).

Needs `langchain-openai` in the environment (§19.6 pins `langchain-openai==1.6.2`,
`langchain-core==1.6.3`, `openai==3.13.0`).

`check()` is defined but not called - it costs seven model calls, so `%run`ing this notebook must
stay free. Run `check()` in a cell to exercise it.

In [ ]:
%run ../setup/config

In [ ]:
%run ./parsing

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# role -> (base URL, model). The only place a role becomes a model name.
ROLE_MODELS = {
    "router": (AI_GATEWAY_URL, MODEL_ROUTER),
    "worker": (AI_GATEWAY_URL, MODEL_ROUTER),
    "synthesizer": (AI_GATEWAY_URL, MODEL_ROUTER),
    "guard_in": (SERVING_URL, MODEL_GUARD_IN),
    "guard_out": (SERVING_URL, MODEL_GUARD_OUT),
    "embedding": (AI_GATEWAY_URL, MODEL_EMBEDDINGS),
}

# §21: reasoning models spent the whole output budget thinking and returned empty content in ~40%
# of calls. Floor every chat call.
MIN_OUTPUT_TOKENS = 300

_token_cache = None


def _token():
    global _token_cache
    if _token_cache is None:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        _token_cache = ctx.apiToken().get()
    return _token_cache


def get_chat_model(role, temperature=None, max_tokens=None):
    if role not in ROLE_MODELS or role == "embedding":
        raise ValueError(f"unknown chat role {role!r}; known: "
                         f"{sorted(r for r in ROLE_MODELS if r != 'embedding')}")
    url, model = ROLE_MODELS[role]
    settings = ROLE_SETTINGS[role]
    cap = max(max_tokens or settings["max_tokens"], MIN_OUTPUT_TOKENS)
    return ChatOpenAI(
        model=model,
        base_url=url,
        api_key=_token(),
        temperature=settings["temperature"] if temperature is None else temperature,
        timeout=MODEL_TIMEOUT_SECONDS,
        # langchain-openai rewrites max_tokens to max_completion_tokens, which the gateway rejects
        # with 400 unknown field (§21). Pass the cap through untouched.
        extra_body={"max_tokens": cap},
    )


def get_embedding_model():
    url, model = ROLE_MODELS["embedding"]
    # check_embedding_ctx_length would try to tokenize for an OpenAI model name that is not one
    return OpenAIEmbeddings(model=model, base_url=url, api_key=_token(),
                            check_embedding_ctx_length=False)

In [ ]:
from pydantic import BaseModel, Field


class _Probe(BaseModel):
    """Flat, every field required - the shape §4.2 requires of every schema."""
    answer: str = Field(description="the capital city")
    confident: bool = Field(description="whether the answer is certain")


def check():
    """Seven live calls: every chat role answers, embeddings return a vector, and one structured
    round trip on the router - which is what settles whether response_format is honoured there
    (agent-design.md §22, open items 10 and 11). Returns a result dict and raises on failure.
    """
    rows = []

    for role in (r for r in ROLE_MODELS if r != "embedding"):
        try:
            text = message_text(get_chat_model(role).invoke(
                [{"role": "user", "content": "Reply with the single word: ready"}]))
            rows.append((role, bool(text.strip()), f"{ROLE_MODELS[role][1]} -> {text.strip()[:60]!r}"))
        except Exception as exc:
            rows.append((role, False, f"{type(exc).__name__}: {exc}"))

    try:
        vector = get_embedding_model().embed_query("What is Databricks?")
        rows.append(("embedding", len(vector) > 0, f"{MODEL_EMBEDDINGS} -> dim {len(vector)}"))
    except Exception as exc:
        rows.append(("embedding", False, f"{type(exc).__name__}: {exc}"))

    unsupported = check_schema_supported(_Probe)
    rows.append(("probe_schema", not unsupported, f"unsupported={unsupported}" if unsupported else "flat"))

    fallback = _Probe(answer="", confident=False)
    result = ask_structured(get_chat_model("router"), "Answer about world capitals.",
                            "What is the capital of France?", _Probe, fallback)
    rows.append(("structured_output", bool(result.answer.strip()),
                 f"{result.answer!r} confident={result.confident}" if result.answer.strip()
                 else "fell back to default - response_format not honoured"))

    for role, ok, detail in rows:
        print(f"{'PASS' if ok else 'FAIL'}  {role:<18}  {detail}")

    failed = [role for role, ok, _ in rows if not ok]
    if failed:
        raise RuntimeError(f"llm/gateway check failed: {failed}")
    return {"ok": True, "rows": rows}